# Bài 7 · Xử lý dữ liệu chuỗi

**Lập trình xử lý dữ liệu (LTXLDL) · 2627-1 · Viện TTNT, UET-VNU**

> 💡 File → **Save a copy in Drive** trước khi sửa.

**Mục tiêu bài học** — sau notebook này, bạn sẽ:

1. Dùng họ `.str` để chuẩn hoá, lọc, sửa, tách cột văn bản và xử lý giá trị thiếu.
2. Đọc–viết được biểu thức chính quy cơ bản (`\d \w \s`, `+ * ?`, `( )`, `|`) và dùng `str.extract`.
3. Biến cột chữ thành **cột tín hiệu** phân tích được và biết quy trình kiểm chứng một biểu thức chính quy.

In [ ]:
import json
import pandas as pd

URL_FULL = ("https://data.insideairbnb.com/chile/rm/santiago/"
            "2026-06-29/data/listings.csv.gz")
df = pd.read_csv(URL_FULL, usecols=["id", "name", "amenities", "price",
                                    "neighbourhood_cleansed"])
df["price_num"] = (df["price"].str.replace("$", "", regex=False)
                              .str.replace(",", "", regex=False)
                              .astype(float))
print(df.shape)
df[["name", "price"]].head(3)

## 1. Họ `.str` — thao tác chuỗi chạy cả cột

In [ ]:
# Chuẩn hoá ba cách viết của cùng một quận
s = pd.Series(["  Ñuñoa ", "ñuñoa", "NUNOA?  "])
print(s.tolist(), "->", s.str.strip().str.lower().tolist())

In [ ]:
# contains: bao nhiêu tên chỗ ở nhắc đến "metro"?
gan_metro = df["name"].str.contains("metro", case=False, na=False)
print(f"{gan_metro.sum()} tên ({gan_metro.mean():.0%})")

In [ ]:
# Với Series kiểu object, contains giữ lại None nếu không khai báo na
s2 = pd.Series(["Depto centro", None, "Casa vista"], dtype=object)
print("Mặc định   :", s2.str.contains("centro").tolist())
print("Có na=False:", s2.str.contains("centro", na=False).tolist())

In [ ]:
# map + explode: chuyển danh sách JSON trong một ô thành nhiều dòng
print(df["amenities"].iloc[0][:90], "…")

tien_nghi = df["amenities"].map(json.loads).explode()
tien_nghi.value_counts().head(10)

Có thể dùng `value_counts().tail(20)` để xem các tiện nghi hiếm. Khi phân tích cột đa trị,
cần kiểm tra cả giá trị phổ biến và giá trị ít xuất hiện.

## 2. Biểu thức chính quy — mô tả mẫu chuỗi

Bài toán mẫu: trích xuất **số phòng ngủ** từ tên chỗ ở ("Modern 1BR Oasis", "Casa 3 dorm centro"…).
Các biến thể khác nhau nhưng chung cấu trúc: *số → khoảng trắng tuỳ ý → từ chỉ phòng ngủ*.

In [ ]:
vi_du = pd.Series([
    "Modern 1BR Oasis",          # khớp: 1
    "Casa 3 dorm centro",        # khớp: 3
    "2 bedrooms near park",      # khớp: 2
    "Depto frente al metro",     # không khớp
    "Habitación año 2024",       # không khớp: 2024 không phải số phòng
])

PAT = r"(\d+)\s*(?:BR|bed|dorm|hab)"
vi_du.str.extract(PAT, expand=False)

Đọc mẫu từng phần: `(\d+)` trích xuất một hoặc nhiều chữ số; `\s*` là khoảng trắng tuỳ ý;
`(?:BR|bed|dorm|hab)` là một trong các từ khoá và không được trả về kết quả.

Trường hợp thứ năm không khớp vì sau số 2024 không có từ khoá phòng ngủ.
**Bộ ví dụ cần chứa cả trường hợp phải khớp và không được khớp** để kiểm chứng biểu thức chính quy.

In [ ]:
# Áp dụng cho cả cột và kiểm đếm
df["so_phong_ngu"] = df["name"].str.extract(PAT, expand=False).astype(float)
print("Số tên tiết lộ số phòng ngủ:", df["so_phong_ngu"].notna().sum())
df.loc[df["so_phong_ngu"].notna(), ["name", "so_phong_ngu"]].head(3)

In [ ]:
# Cùng bài giá tiền — dùng biểu thức chính quy để kiểm tra đúng định dạng
gia_trich = (df["price"]
             .str.extract(r"\$([\d,]+(?:\.\d+)?)", expand=False)
             .str.replace(",", "", regex=False)
             .astype(float))
print("Khớp với cách replace:", gia_trich.equals(df["price_num"]))

## 3. Cột chữ → cột tín hiệu

In [ ]:
df["gan_metro"] = gan_metro
df["co_view"] = df["name"].str.contains(r"view|vista", case=False, na=False)
df["co_dau_tbn"] = df["name"].str.contains(r"ción|ñ|á|é|í|ó|ú", case=False, na=False)

df[["gan_metro", "co_view", "co_dau_tbn"]].mean().round(3)

In [ ]:
# So sánh giá theo tín hiệu tên có nhắc đến metro
df.groupby("gan_metro")["price_num"].agg(["median", "size"]).round(1)

Nhóm tên có chữ "metro" có giá trung vị thấp hơn. Khác biệt này chưa cho biết nguyên nhân;
cần kiểm tra thêm loại phòng và khu vực trước khi diễn giải. Trích xuất không đồng nghĩa với kết luận.

## 4. Bài tập tại lớp

### Bài 1 — Từ điển tiện nghi

Dùng `tien_nghi` (mục 1): tính **tỷ lệ chỗ ở** có (a) máy giặt (`Washer`), (b) điều hoà
(chú ý: có nhiều biến thể "Air conditioning", "AC unit"… — dùng `contains`), (c) chỗ đỗ xe
(`parking`, cũng nhiều biến thể). Gợi ý: quay về mức từng chỗ ở bằng
`df["amenities"].str.contains(...)` cho nhanh.

In [ ]:
# TODO Bài 1:
for nhan, pat in [("Máy giặt", r"Washer"), ("Điều hoà", r"[Aa]ir condition|AC unit"),
                  ("Đỗ xe", r"parking")]:
    ty_le = df["amenities"].str.contains(pat, case=False, na=False).mean()
    print(f"{nhan:10}: {ty_le:.0%}")

### Bài 2 — Viết biểu thức chính quy và kiểm chứng

Nhiều tên phòng ghi diện tích: "Depto 45 m2", "Studio 30m²", "80 M2 luxury". Hãy:

1. Tự viết 5 ví dụ (3 khớp, 2 không — ví dụ "Torre 2024" không được khớp).
2. Viết biểu thức chính quy để trích **con số diện tích**; chạy trên 5 ví dụ.
3. Áp lên `df["name"]`, đếm số chỗ ở có ghi diện tích, xem 5 dòng khớp đầu tiên.

In [ ]:
# TODO Bài 2 (khung gợi ý — hoàn thiện mẫu):
PAT_M2 = r"(\d+)\s*[mM](?:2|²)\b"
test = pd.Series(["Depto 45 m2", "Studio 30m²", "80 M2 luxury", "Torre 2024", "Casa centro"])
print(test.str.extract(PAT_M2, expand=False).tolist())

df["dien_tich"] = df["name"].str.extract(PAT_M2, expand=False).astype(float)
print("Số chỗ ở có ghi diện tích:", df["dien_tich"].notna().sum())
df.loc[df["dien_tich"].notna(), ["name", "dien_tich"]].head(5)

### Bài 3 — Chuẩn hoá để đếm đúng

`neighbourhood_cleansed` của Santiago khá sạch. Ô lệnh dưới tạo cột `quan_ban` với nhiều
cách viết; hãy chuẩn hoá thành `quan_chuan` sao cho `value_counts()` **khớp** với bản gốc.

In [ ]:
import numpy as np
rng = np.random.default_rng(7)
bien_the = {0: lambda s: s, 1: lambda s: s.upper(), 2: lambda s: "  " + s + " ",
            3: lambda s: s.lower()}
df["quan_ban"] = [bien_the[k](s) for k, s in
                  zip(rng.integers(0, 4, len(df)), df["neighbourhood_cleansed"])]

# TODO: chuẩn hoá quan_ban về dạng gốc, rồi đối chiếu (gợi ý: strip + title)
df["quan_chuan"] = df["quan_ban"].str.strip().str.title()
goc = df["neighbourhood_cleansed"].str.title().value_counts()
chuan = df["quan_chuan"].value_counts()
print("Khớp hoàn toàn:", goc.equals(chuan))

## 5. Bài tập về nhà — Xây dựng từ điển khía cạnh

Bài 11 sẽ so sánh LLM với phương pháp đối chứng (baseline) bằng biểu thức chính quy/từ khoá:

1. Chọn 4 khía cạnh của chỗ ở (vị trí, sạch sẽ, chủ nhà, giá trị…).
2. Với mỗi khía cạnh, viết mẫu từ khoá **hai ngôn ngữ** (Anh + Tây Ban Nha) — như
   `ASPECT_KEYWORDS` bạn sẽ gặp ở Bài 11.
3. Áp lên cột `name` (tạm thay cho bảng `reviews`): tỷ lệ nhắc đến từng khía cạnh?
4. Kiểm tra 10 dòng khớp của mỗi khía cạnh; ghi lại 2–3 trường hợp khớp nhầm và nguyên nhân.

Bước 4 là phân tích lỗi của baseline và sẽ được dùng lại ở Bài 11.

In [ ]:
RUN_CHALLENGE = False

if RUN_CHALLENGE:
    ASPECT_KEYWORDS = {
        "vi_tri": r"location|central|centro|metro|ubicaci",
        # TODO: thêm 3 khía cạnh nữa
    }
    ...

---

## Tóm tắt bài học

| Nội dung chính | Vì sao quan trọng |
|---|---|
| `strip`/`lower` trước khi đếm; `na=False` khi `contains` | Đếm đúng và xử lý rõ giá trị thiếu |
| `explode` cho ô đa trị (`amenities`) | Đưa từng giá trị về một dòng để thống kê |
| Biểu thức chính quy mô tả mẫu; `str.extract` + nhóm `( )` | Trích xuất dữ liệu từ văn bản tự do |
| Kiểm chứng biểu thức chính quy bằng ví dụ khớp và không khớp | Phát hiện mẫu quá rộng hoặc quá hẹp |
| Cột chuỗi → cột tín hiệu → `groupby` | Tạo baseline cho phần LLM ở Bài 11 |

**Bài sau:** dữ liệu thời gian — datetime, resample và so sánh theo thời điểm.